# QUADAS-2 analysis

1. **Code part**: imports, file paths, data prep.
2. **Run part**: the cells that create the intra- and inter-observer QUADAS-2 figures.

Each figure run prints the number of articles and article IDs in each category: Low risk, High risk, and Unclear.

## Code part

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
DATA_EXTRACTION_FILE = Path(r"PATH/TO/DATA.xlsx")
QUADAS_FILE = Path(r"PATH/TO/QUADAS/SCORING.xlsx")
OUTPUT_DIR = Path(r"PATH/OUTPUT/FOLDER")

GASTRIC_COHORT_COLUMN = "Gastric_Cohort"
ARTICLE_ID_COLUMN = "article nr"
LABEL_COLUMN = "study_ID"
DOMAIN_COLUMN = "DOMAIN"
STUDY_COLUMN_START = 2

CATEGORY_COLORS = {
    "Low risk": "#3a7d1c",
    "Unclear": "#ffd92f",
    "High risk": "#d7191c",
}

In [ ]:
def load_quadas_data():
    """Load extraction and QUADAS-2 spreadsheets."""
    extraction_df = pd.read_excel(DATA_EXTRACTION_FILE, header=1)
    intra_df = pd.read_excel(QUADAS_FILE, sheet_name="QUADAS_intra")
    inter_df = pd.read_excel(QUADAS_FILE, sheet_name="QUADAS_inter")
    return extraction_df, intra_df, inter_df


def get_gastric_article_ids(extraction_df):
    """Return article IDs marked as gastric cohort articles."""
    return extraction_df.loc[
        extraction_df[GASTRIC_COHORT_COLUMN] == 1,
        ARTICLE_ID_COLUMN,
    ].tolist()

def filter_to_article_ids(quadas_df, article_ids):
    """Keep QUADAS metadata columns plus the selected article ID columns."""
    article_id_set = set(article_ids) # gastric papers.
    metadata_cols = quadas_df.iloc[:, :STUDY_COLUMN_START]
    study_cols = [
        column for column in quadas_df.columns[STUDY_COLUMN_START:]
        if column in article_id_set
    ]
    return pd.concat([metadata_cols, quadas_df.loc[:, study_cols]], axis=1)

def prepare_quadas_data():
    """Load all input files and return gastric-cohort QUADAS dataframes."""
    extraction_df, intra_df, inter_df = load_quadas_data()
    gastric_ids = get_gastric_article_ids(extraction_df)

    intra_gastric_df = filter_to_article_ids(intra_df, gastric_ids)
    inter_gastric_df = filter_to_article_ids(inter_df, gastric_ids)

    print(f"Gastric cohort articles: {len(gastric_ids)}")
    print(gastric_ids)
    print(f"Intra QUADAS articles retained: {len(intra_gastric_df.columns[STUDY_COLUMN_START:])}")
    print(list(intra_gastric_df.columns[STUDY_COLUMN_START:]))
    print(f"Inter QUADAS articles retained: {len(inter_gastric_df.columns[STUDY_COLUMN_START:])}")
    print(list(inter_gastric_df.columns[STUDY_COLUMN_START:]))

    return intra_gastric_df, inter_gastric_df


In [ ]:
def _category_mask(values, category):
    if category == "Unclear":
        return values.isna()
    if category == "Low risk":
        return values == 0
    if category == "High risk":
        return values == 1
    raise ValueError(f"Unknown category: {category}")

def summarize_quadas(quadas_label_df):
    """Summarise percentages per domain for one QUADAS-2 label."""
    study_cols = list(quadas_label_df.columns[STUDY_COLUMN_START:])
    total_articles = len(study_cols)
    rows = []

    for _, row in quadas_label_df.iterrows():
        values = row[study_cols]
        summary_row = {DOMAIN_COLUMN: row[DOMAIN_COLUMN]}

        for category in ["Low risk", "Unclear", "High risk"]:
            count = int(_category_mask(values, category).sum())
            summary_row[category] = count / total_articles * 100 if total_articles else 0

        rows.append(summary_row)

    return pd.DataFrame(rows)


def category_article_details(quadas_label_df):
    """Return counts and article IDs per domain and category for one plotted label."""
    study_cols = list(quadas_label_df.columns[STUDY_COLUMN_START:])
    rows = []

    for _, row in quadas_label_df.iterrows():
        values = row[study_cols]

        for category in ["Low risk", "High risk", "Unclear"]:
            ids = [
                article_id for article_id in study_cols
                if _category_mask(values, category).loc[article_id]
            ]
            rows.append({
                "Domain": row[DOMAIN_COLUMN],
                "Category": category,
                "N articles": len(ids),
                "Article IDs": ids,
            })

    return pd.DataFrame(rows)

def print_category_article_details(details_df, title):
    """Print article counts and IDs for the figure represented by details_df."""
    print(f"\n{title}")
    print("=" * len(title))

    for domain, domain_df in details_df.groupby("Domain", sort=False):
        print(f"\n{domain}")
        for _, row in domain_df.iterrows():
            ids = row["Article IDs"]
            id_text = ", ".join(map(str, ids)) if ids else "None"
            print(f"- {row['Category']}: {row['N articles']} article(s): {id_text}")


In [ ]:
def plot_quadas(summary_df, title):
    """Create and save a QUADAS-2 stacked horizontal bar chart."""
    fig, ax = plt.subplots(figsize=(8, 4))

    left = np.zeros(len(summary_df))
    for category in ["Low risk", "Unclear", "High risk"]:
        ax.barh(
            summary_df[DOMAIN_COLUMN],
            summary_df[category],
            left=left,
            color=CATEGORY_COLORS[category],
            label=category,
        )
        left += summary_df[category].to_numpy()

    ax.set_xlim(0, 100)
    ax.set_xlabel("Percentage of studies")
    ax.set_title(title)
    ax.invert_yaxis()
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))

    plt.tight_layout()

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    safe_title = (
        title.replace(":", "")
        .replace("/", "-")
        .replace("\\", "-")
        .replace(" ", "_")
    )
    
    png_file = OUTPUT_DIR / f"QUADAS_fig_{safe_title}.png"
    svg_file = OUTPUT_DIR / f"QUADAS_fig_{safe_title}.svg"
    
    plt.savefig(png_file, dpi=300, bbox_inches="tight")
    plt.savefig(svg_file, bbox_inches="tight")
    
    print(f"Saved PNG: {png_file}")
    print(f"Saved SVG: {svg_file}")
    
    plt.show()

In [ ]:
def quadas_figure(quadas_df, label, dataset_name):
    """Print category details, display summary table, and create one QUADAS-2 figure."""
    quadas_label_df = quadas_df[quadas_df[LABEL_COLUMN] == label]
    title = f"QUADAS-2: {label} by domain-{dataset_name}"

    summary_df = summarize_quadas(quadas_label_df)
    details_df = category_article_details(quadas_label_df)

    print_category_article_details(details_df, title)
    display(summary_df)
    plot_quadas(summary_df, title)

    return summary_df, details_df


## Run part

In [ ]:
df_intra_gastric, df_inter_gastric = prepare_quadas_data()

### Intra-observer figures


In [ ]:
intra_risk_summary, intra_risk_details = quadas_figure(
    df_intra_gastric,
    label="Risk of bias",
    dataset_name="intra",
)

intra_applicability_summary, intra_applicability_details = quadas_figure(
    df_intra_gastric,
    label="Concerns regarding applicability",
    dataset_name="intra",
)


### Inter-observer figures


In [ ]:
inter_risk_summary, inter_risk_details = quadas_figure(
    df_inter_gastric,
    label="Risk of bias",
    dataset_name="inter",
)

inter_applicability_summary, inter_applicability_details = quadas_figure(
    df_inter_gastric,
    label="Concerns regarding applicability",
    dataset_name="inter",
)
